# Taller B3-T4 - Redes Neuronales para Forecasting
## Ventana entrada: 1 dia | Ventana salida: 5 dias

- **Parte 1 - Competicion**: entrenar y comparar modelos sobre log-retornos en bruto.
- Se sigue la estructura del cuaderno `ent90_sal30`, adaptando `Conv1D` a `kernel=1` porque la ventana de entrada tiene un solo paso temporal.


In [1]:
VENTANA_ENTRADA = 1
VENTANA_SALIDA = 5

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from IPython.display import display
from sklearn.linear_model import LinearRegression
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

from utilidades.carga_datos import cargar_retornos, create_time_series_data, dividir_datos, aplanar_X
from utilidades.modelos import construir_dense, construir_recurrente, construir_conv1d, construir_mixto
from utilidades.evaluacion import evaluar_modelo, evaluar_sklearn, evaluar_buyhold, guardar_resultados
from utilidades.graficos import graficar_convergencia, graficar_barras_mae

SEED = 42
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

EPOCHS = 80
BATCH_SIZE = 128
CALLBACKS = [
    EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_delta=1e-6, min_lr=1e-7, verbose=1),
]


---
# PARTE 1 - Competicion
Modelos sobre log-retornos en bruto. Metrica: MAE medio sobre 23 activos.

## 1.1 Carga de datos

In [3]:
retornos = cargar_retornos()
X, y = create_time_series_data(retornos, VENTANA_ENTRADA, VENTANA_SALIDA)
print(f'X: {X.shape} | y: {y.shape}')

X_train, X_val, X_test, y_train, y_val, y_test = dividir_datos(X, y)
X_train_plano = aplanar_X(X_train)
X_val_plano = aplanar_X(X_val)
X_test_plano = aplanar_X(X_test)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Entrada plana: {X_train_plano.shape}')


X: (16185, 1, 23) | y: (16185, 23)
Train: (13837, 1, 23)  Val: (729, 1, 23)  Test: (1619, 1, 23)
Entrada plana: (13837, 23)


## 1.2 Baselines

In [4]:
reg_lineal = LinearRegression()
reg_lineal.fit(X_train_plano, y_train)
resultado_lineal = evaluar_sklearn(
    reg_lineal, X_train_plano, y_train,
    X_val_plano, y_val, X_test_plano, y_test, nombre='Lineal')
resultado_bah = evaluar_buyhold(y_train, y_val, y_test)

display(pd.DataFrame([resultado_lineal, resultado_bah]).set_index('modelo').round(6))


,mae_train,mae_val,mae_test,n_params
modelo,,,,
Lineal,0.005386,0.004303,0.005594,0
BuyAndHold,0.005395,0.004289,0.005581,0


## 1.3 Entrenamiento y evaluacion

In [5]:
def entrenar_y_evaluar(nombre, constructor, X_tr, X_v, X_ts):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    modelo = constructor()
    modelo.summary()
    hist = modelo.fit(
        X_tr, y_train,
        validation_data=(X_v, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=CALLBACKS,
        verbose=1,
    )
    graficar_convergencia(hist, nombre)
    resultado = evaluar_modelo(
        modelo, X_tr, y_train, X_v, y_val, X_ts, y_test, nombre=nombre)
    print(resultado)
    return modelo, hist, resultado


### Dense (MLP)

In [6]:
modelo_dense, hist_dense, resultado_dense = entrenar_y_evaluar(
    'Dense',
    lambda: construir_dense(X_train_plano.shape[1], y_train.shape[1]),
    X_train_plano, X_val_plano, X_test_plano,
)


C:\Users\ORDENADOR\Taller4_NN\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 23)             │         2,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,007 (164.09 KB)

 Trainable params: 42,007 (164.09 KB)

...


Epoch 24: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.


109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0054 - val_loss: 0.0043 - learning_rate: 2.5000e-04


Epoch 24: early stopping


Restoring model weights from the end of the best epoch: 14.


{'modelo': 'Dense', 'mae_train': 0.005386970634489945, 'mae_val': 0.004296201732389664, 'mae_test': 0.005593106881045149, 'n_params': 42007}


### Recurrente (LSTM)

In [7]:
modelo_lstm, hist_lstm, resultado_lstm = entrenar_y_evaluar(
    'LSTM',
    lambda: construir_recurrente(X_train.shape[1:], y_train.shape[1]),
    X_train, X_val, X_test,
)


C:\Users\ORDENADOR\Taller4_NN\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        22,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │         1,495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,023 (93.84 KB)

 Trainable params: 24,023 (93.84 KB)

...


Epoch 10: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.


109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0054 - val_loss: 0.0043 - learning_rate: 5.0000e-04


Epoch 10: early stopping


Restoring model weights from the end of the best epoch: 1.


{'modelo': 'LSTM', 'mae_train': 0.005442489576681451, 'mae_val': 0.004357895904355638, 'mae_test': 0.005630862043592646, 'n_params': 24023}


### Conv1D

In [8]:
# Con VENTANA_ENTRADA=1, kernel=3 no cabe en la dimension temporal.
# Usamos el constructor existente con kernel=1, sin modificar modelos.py.
modelo_conv, hist_conv, resultado_conv = entrenar_y_evaluar(
    'Conv1D',
    lambda: construir_conv1d(X_train.shape[1:], y_train.shape[1], kernel=1),
    X_train, X_val, X_test,
)


C:\Users\ORDENADOR\Taller4_NN\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Conv1D"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1, 64)          │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 1, 32)          │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │           759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,375 (17.09 KB)

 Trainable params: 4,375 (17.09 KB)

...


Epoch 24: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.


109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0054 - val_loss: 0.0043 - learning_rate: 1.2500e-04


Epoch 24: early stopping


Restoring model weights from the end of the best epoch: 14.


{'modelo': 'Conv1D', 'mae_train': 0.005393472015261946, 'mae_val': 0.004293630023224049, 'mae_test': 0.005586152761166267, 'n_params': 4375}


### Mixto (Conv1D + LSTM)

In [9]:
modelo_mixto, hist_mixto, resultado_mixto = entrenar_y_evaluar(
    'Mixto',
    lambda: construir_mixto(X_train.shape[1:], y_train.shape[1]),
    X_train, X_val, X_test,
)


Model: "Mixto"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 23)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 1, 64)          │         4,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │         1,495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,999 (152.34 KB)

 Trainable params: 38,999 (152.34 KB)

 Non-trainable params: 0 (0.00 B)

...


Epoch 10: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.


109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0054 - val_loss: 0.0043 - learning_rate: 5.0000e-04


Epoch 10: early stopping


Restoring model weights from the end of the best epoch: 1.


{'modelo': 'Mixto', 'mae_train': 0.005427570338493019, 'mae_val': 0.004340373808908125, 'mae_test': 0.005603475797318738, 'n_params': 38999}


## 1.4 Resumen de competicion y guardado

In [10]:
resultados_competicion = [
    resultado_lineal, resultado_bah,
    resultado_dense, resultado_lstm,
    resultado_conv, resultado_mixto,
]

graficar_barras_mae(resultados_competicion, VENTANA_ENTRADA, VENTANA_SALIDA)
guardar_resultados(
    resultados_competicion,
    VENTANA_ENTRADA,
    VENTANA_SALIDA,
    seccion='competicion',
)

df_resultados = pd.DataFrame(resultados_competicion).set_index('modelo').round(6)
display(df_resultados)


Resultados [competicion] guardados en: ../resultados/metricas/ent01_sal05.json


C:\Users\ORDENADOR\Taller4_NN\cuadernos\..\utilidades\graficos.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mae_train,mae_val,mae_test,n_params
modelo,,,,
Lineal,0.005386,0.004303,0.005594,0
BuyAndHold,0.005395,0.004289,0.005581,0
Dense,0.005387,0.004296,0.005593,42007
LSTM,0.005442,0.004358,0.005631,24023
Conv1D,0.005393,0.004294,0.005586,4375
Mixto,0.005428,0.004340,0.005603,38999
